[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tasmia008/Complete_Guidelines_LLM_FineTuning/blob/main/02_concept_supervised_vs_selfsupervised.ipynb)

# Concept — supervised vs. self-supervised fine-tuning

> **Fine-tuning series — 02 of 26.** A standalone notebook in a set; see `00_README.md` for the full index and order. Conceptual/reference — no GPU setup needed.


## Concept — supervised vs. self-supervised fine-tuning

Everything above is *fine-tuning*; the deeper split is **where the training target
comes from.**

**Supervised fine-tuning (SFT).** Targets are *externally provided labels* — a human
or curated process decided the correct output. You train on (input → known answer)
pairs and the loss compares the prediction to that given target. Image labels,
segmentation masks, (prompt → desired response) instruction pairs: all supervised,
because the answer had to be supplied from outside the data.

**Self-supervised fine-tuning (SSL).** No external labels — the target is
*manufactured from the unlabeled data itself* via a pretext task: mask tokens and
predict them, corrupt an image and reconstruct it, hold out a region and predict its
latent (JEPA, MAE, DINO, contrastive). When people say "self-supervised fine-tuning"
they usually mean **continued / domain-adaptive pretraining**: keep running the
self-supervised objective on a new domain's *unlabeled* corpus to adapt the
representations before any labeled task.

**The subtle point.** The *objective* can be identical in both — provenance of the
target is what makes it supervised or not. LLM pretraining and instruction-SFT both
use next-token prediction; in pretraining the next token is just whatever followed in
a scraped document (self-supervised), while in SFT it belongs to a curated answer a
human wrote (supervised). Same loss, different source of truth.

They form a **pipeline, not a competition** — and the middle (self-supervised)
stage matters most exactly when labels are scarce, e.g. medical imaging: unlabeled
scans are abundant, expert annotation is expensive, so adapting representations
self-supervised first lets the supervised stage do more with fewer labels.

In [ ]:
# Pipeline: self-supervised pretraining -> self-supervised fine-tuning -> SFT
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(12, 4.6))
ax.set_xlim(0, 12); ax.set_ylim(0, 5); ax.axis("off")
SSL, SFT, ssl_fill, sft_fill = "#5B8FF9", "#F6BD16", "#EAF1FE", "#FEF6E3"
stages = [
    (0.4, "1. Self-supervised\npretraining", "data: huge UNLABELED corpus",
     "objective: pretext task\n(next-token / masking / JEPA)", "→ general representations", SSL, ssl_fill),
    (4.3, "2. Self-supervised\nfine-tuning", "data: UNLABELED domain data",
     "= continued / domain-\nadaptive pretraining", "→ domain-adapted reps  (optional)", SSL, ssl_fill),
    (8.2, "3. Supervised\nfine-tuning", "data: LABELED task pairs",
     "objective: match the\nprovided target", "→ task model", SFT, sft_fill),
]
w, h, y = 3.3, 2.7, 1.4
for x, title, data, obj, out, edge, fill in stages:
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.06,rounding_size=0.18",
                                linewidth=2, edgecolor=edge, facecolor=fill))
    ax.text(x+w/2, y+h-0.45, title, ha="center", va="center", fontsize=11.5, fontweight="bold", color="#1f2d3d")
    ax.text(x+w/2, y+h-1.25, data, ha="center", va="center", fontsize=9, color="#314155")
    ax.text(x+w/2, y+h-1.95, obj, ha="center", va="center", fontsize=8.5, color="#52606d")
    ax.text(x+w/2, y+0.18, out, ha="center", va="center", fontsize=8.5, style="italic", color=edge)
for x0 in (3.8, 7.7):
    ax.add_patch(FancyArrowPatch((x0, y+h/2), (x0+0.45, y+h/2), arrowstyle="-|>",
                 mutation_scale=18, linewidth=2, color="#8893a0"))
ax.plot([0.4, 7.6], [0.75, 0.75], color=SSL, linewidth=3, solid_capstyle="round")
ax.text(4.0, 0.4, "SELF-SUPERVISED — target made from the data itself",
        ha="center", va="center", fontsize=9, color=SSL, fontweight="bold")
ax.plot([8.2, 11.5], [0.75, 0.75], color=SFT, linewidth=3, solid_capstyle="round")
ax.text(9.85, 0.4, "SUPERVISED — target = external label",
        ha="center", va="center", fontsize=9, color="#B8860B", fontweight="bold")
ax.text(6.0, 4.65, "Same objective can appear in both — what differs is where the target comes from",
        ha="center", va="center", fontsize=9.5, color="#52606d")
fig.tight_layout(); plt.show()